In [1]:
pip install kagglehub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import kagglehub
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")
print(path)

c:\Users\excel\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\excel\.cache\kagglehub\datasets\clmentbisaillon\fake-and-real-news-dataset\versions\1


In [3]:
import nltk
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\excel\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\excel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\excel\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\excel\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\excel\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

In [4]:
import pandas as pd
import os

print(os.listdir(path))

['Fake.csv', 'True.csv']


In [5]:
fake = pd.read_csv(os.path.join(path, "Fake.csv"))
true = pd.read_csv(os.path.join(path, "True.csv"))

In [6]:
fake["label"] = 0
true["label"] = 1

In [7]:
df = pd.concat([fake, true], ignore_index=True)
print(df.head())
print(df.shape)
print(df["label"].value_counts())

                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  label  
0  December 31, 2017      0  
1  December 31, 2017      0  
2  December 30, 2017      0  
3  December 29, 2017      0  
4  December 25, 2017      0  
(44898, 5)
label
0    23481
1    21417
Name: count, dtype: int64


In [8]:
# Remove duplicate and empty articles
df = df.drop_duplicates(subset="text")
df = df[df["text"].str.strip() != ""]

# Remove the "CITY (Reuters) -" prefix, which would leak the label
df["text"] = df["text"].str.replace(r"^[^()]{0,100}\(Reuters\)\s*-?\s*", "", regex=True)

# Use a random sample so preprocessing finishes in reasonable time
df = df.sample(n=5000, random_state=42).reset_index(drop=True)
print(df["label"].value_counts())

label
1    2741
0    2259
Name: count, dtype: int64


In [9]:
df

,title,text,subject,date,label
0,Kenya watchdog says investigating police over ...,(This September 29 has been corrected to fix ...,worldnews,"September 29, 2017",1
1,Obama asked Republican leaders to offer top co...,President Barack Obama asked Republican congre...,politicsNews,"March 1, 2016",1
2,"Kosovo parties sign deal to form government, e...",Kosovo s center-right coalition led by the Dem...,worldnews,"September 4, 2017",1
3,(VIDEO) HILLARY CLINTON JOKES ABOUT THE FBI IN...,Nice that she s making fun of the ability to h...,politics,"Aug 14, 2015",0
4,Chicago immigrant claims U.S. retaliated over ...,A Chicago activist sued U.S. immigration autho...,politicsNews,"May 25, 2016",1
...,...,...,...,...,...
4995,Labor activists target Midwest politicians opp...,U.S. activists plan protests in up to 400 citi...,politicsNews,"September 2, 2017",1
4996,City Of Chicago Just Told Trump To Go F*ck Hi...,Donald Trump s giant tower in Chicago now has ...,News,"July 3, 2017",0
4997,Factbox: Who's in? Who's out? China's Communis...,China s ruling Communist Party announced a new...,worldnews,"October 24, 2017",1
4998,LEGENDARY DIRECTOR Shocks Liberal Elites With ...,It s a rare occasion when a Hollywood legend d...,politics,"Dec 29, 2017",0


In [10]:
df_text = df["text"].str.lower()

In [11]:
df_text

0        (this september 29 has been corrected to fix ...
1       president barack obama asked republican congre...
2       kosovo s center-right coalition led by the dem...
3       nice that she s making fun of the ability to h...
4       a chicago activist sued u.s. immigration autho...
                              ...                        
4995    u.s. activists plan protests in up to 400 citi...
4996    donald trump s giant tower in chicago now has ...
4997    china s ruling communist party announced a new...
4998    it s a rare occasion when a hollywood legend d...
4999    residents of south sudan s capital say they ar...
Name: text, Length: 5000, dtype: str

In [12]:
# Remove Punctuations:
import re, string
df_text = df_text.str.replace(r"[’'‘]", "", regex=True)   # apostrophes: don't -> dont
df_text = df_text.str.replace(rf"[{re.escape(string.punctuation)}“”—–]", " ", regex=True)

In [13]:
# Tokenization:

from nltk.tokenize import word_tokenize
df_text_token = df_text.apply(word_tokenize)

In [14]:
df_text_token

0       [this, september, 29, has, been, corrected, to...
1       [president, barack, obama, asked, republican, ...
2       [kosovo, s, center, right, coalition, led, by,...
3       [nice, that, she, s, making, fun, of, the, abi...
4       [a, chicago, activist, sued, u, s, immigration...
                              ...                        
4995    [u, s, activists, plan, protests, in, up, to, ...
4996    [donald, trump, s, giant, tower, in, chicago, ...
4997    [china, s, ruling, communist, party, announced...
4998    [it, s, a, rare, occasion, when, a, hollywood,...
4999    [residents, of, south, sudan, s, capital, say,...
Name: text, Length: 5000, dtype: object

In [15]:
# Stop words Removal:
import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")
stop_words = set(stopwords.words('english'))
stop_words.add("reuters")
df_clean_token = df_text_token.apply(lambda words: [word for word in words if word not in stop_words])

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\excel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [16]:
df_clean_token

0       [september, 29, corrected, fix, date, election...
1       [president, barack, obama, asked, republican, ...
2       [kosovo, center, right, coalition, led, democr...
3       [nice, making, fun, ability, messages, disappe...
4       [chicago, activist, sued, u, immigration, auth...
                              ...                        
4995    [u, activists, plan, protests, 400, cities, ac...
4996    [donald, trump, giant, tower, chicago, appropr...
4997    [china, ruling, communist, party, announced, n...
4998    [rare, occasion, hollywood, legend, defends, p...
4999    [residents, south, sudan, capital, say, collec...
Name: text, Length: 5000, dtype: object

In [17]:
# Lemmetization:

from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

df_clean_token = df_clean_token.apply(lambda x: [lemmatizer.lemmatize(word) for word in x])
df_clean_token

0       [september, 29, corrected, fix, date, election...
1       [president, barack, obama, asked, republican, ...
2       [kosovo, center, right, coalition, led, democr...
3       [nice, making, fun, ability, message, disappea...
4       [chicago, activist, sued, u, immigration, auth...
                              ...                        
4995    [u, activist, plan, protest, 400, city, across...
4996    [donald, trump, giant, tower, chicago, appropr...
4997    [china, ruling, communist, party, announced, n...
4998    [rare, occasion, hollywood, legend, defends, p...
4999    [resident, south, sudan, capital, say, collect...
Name: text, Length: 5000, dtype: object

In [18]:
# POS tagging:
from nltk import pos_tag
Data_POS = df_clean_token.head(1000).apply(pos_tag)
Data_POS.head()

0    [(september, VB), (29, CD), (corrected, JJ), (...
1    [(president, NN), (barack, NN), (obama, NN), (...
2    [(kosovo, NN), (center, NN), (right, JJ), (coa...
3    [(nice, JJ), (making, VBG), (fun, JJ), (abilit...
4    [(chicago, NNS), (activist, VBP), (sued, VBN),...
Name: text, dtype: object

In [19]:
# Feature Engineering:
# TF-IDF vectorisation:

from sklearn.feature_extraction.text import TfidfVectorizer
Data = df_clean_token.apply(lambda words: " ".join(words))
tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(Data)
tfidf_df = pd.DataFrame(
    tfidf_matrix[:5].toarray(),
    columns=tfidf.get_feature_names_out()
)
tfidf_df

,00,000,10,100,11,12,120,13,130,14,...,youtube,ypg,zarrab,zealand,zero,zika,zimbabwe,zone,zor,zuma
0,0.0,0.00000,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.00000,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.04958,0.0,0.0,0.064093,0.0,0.19353,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.00000,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.00000,0.0,0.0,0.000000,0.0,0.00000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
# Unigram 
# Bigram,
# N gram. 

def n_gram_generation(data, n=1):
    if isinstance(data, pd.Series):
        return [[" ".join(s.split()[i:i+n]) for i in range(len(s.split())-n+1)] for s in data]
    else:
        print("Only accepting Series")

In [21]:
print("Unigrams:", n_gram_generation(Data.head(1), n=1)[0][:10])
print("Bigrams: ", n_gram_generation(Data.head(1), n=2)[0][:10])
print("Trigrams:", n_gram_generation(Data.head(1), n=3)[0][:10])

Unigrams: ['september', '29', 'corrected', 'fix', 'date', 'election', 'paragraph', '3', 'nairobi', 'kenyan']
Bigrams:  ['september 29', '29 corrected', 'corrected fix', 'fix date', 'date election', 'election paragraph', 'paragraph 3', '3 nairobi', 'nairobi kenyan', 'kenyan government']
Trigrams: ['september 29 corrected', '29 corrected fix', 'corrected fix date', 'fix date election', 'date election paragraph', 'election paragraph 3', 'paragraph 3 nairobi', '3 nairobi kenyan', 'nairobi kenyan government', 'kenyan government watchdog']


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(Data, y, test_size=0.2, random_state=42, stratify=y)
vec = TfidfVectorizer(max_features=5000)
X_train_vec = vec.fit_transform(X_train)
X_test_vec = vec.transform(X_test)
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

In [23]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Fake", "Real"]))

Accuracy : 0.968
Precision: 0.9708029197080292
Recall   : 0.9708029197080292
F1-score : 0.9708029197080292

Confusion Matrix:
 [[436  16]
 [ 16 532]]

Classification Report:
               precision    recall  f1-score   support

        Fake       0.96      0.96      0.96       452
        Real       0.97      0.97      0.97       548

    accuracy                           0.97      1000
   macro avg       0.97      0.97      0.97      1000
weighted avg       0.97      0.97      0.97      1000



<hr>

<h3 style="text-align:center; color:green;">
    End of Assignment
</h3>

<hr>